# EDA

Хакатон ChemAI: Predict the Cure.
В этом ноутбуке выполняется разведочный анализ данных для второй группы (Электронные и зарядовые характеристики) молекулярных дескрипторов.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy import stats
from sklearn.feature_selection import VarianceThreshold
from IPython.display import display
from sklearn.cluster import KMeans
from scipy.stats import f_oneway
from sklearn.preprocessing import OneHotEncoder
warnings.filterwarnings('ignore')
plt.style.use('ggplot')
sns.set_palette('viridis')
%matplotlib inline

SEED = 42
np.random.seed(SEED)

## Загрузка данных

Файлы `train.csv`, `test.csv` должны находиться в папке `data/`.


In [2]:
import os
from google.colab import files

# Create 'data' directory if it doesn't exist
if not os.path.exists('data'):
    os.makedirs('data')

print("Please upload 'train.csv' and 'test.csv'.")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f'User uploaded file "{filename}" with length {len(uploaded[filename])} bytes')
    # Move the uploaded file to the 'data/' directory
    os.rename(filename, os.path.join('data', filename))
    print(f'Moved {filename} to data/{filename}')


Please upload 'train.csv' and 'test.csv'.


KeyboardInterrupt: 

In [ ]:
train = pd.read_csv('data/train_.csv')
test = pd.read_csv('data/test_.csv')

train.rename(columns={'IC50, mM': 'IC50', 'CC50, mM': 'CC50'}, inplace=True)

TARGETS = ['IC50', 'CC50', 'SI']


**Вывод:**  
Данные успешно загружены. Обучающая выборка содержит 751 молекулу и 214 столбцов (включая индекс, таргеты и дескрипторы).  
Тестовая выборка — 250 молекул с теми же дескрипторами без таргетов.

## Вспомогательные функции (единые для всех групп)




In [ ]:
def display_columns_info(df, columns=None):
    """Подробная статистика по колонкам: тип, пропуски, уникальные значения, мин/макс/среднее/стд."""
    if columns is None:
        columns = df.columns.tolist()
    info_list = []
    for col in columns:
        info_list.append({
            'column': col,
            'dtype': df[col].dtype,
            'missing': df[col].isnull().sum(),
            'missing_pct': df[col].isnull().sum() / len(df) * 100,
            'unique': df[col].nunique(),
            'min': df[col].min() if df[col].dtype in ['float64', 'int64'] else None,
            'max': df[col].max() if df[col].dtype in ['float64', 'int64'] else None,
            'mean': df[col].mean() if df[col].dtype in ['float64', 'int64'] else None,
            'std': df[col].std() if df[col].dtype in ['float64', 'int64'] else None
        })
    info_df = pd.DataFrame(info_list)
    display(info_df)

def interpret_correlation_strength(corr_coef):
    """Интерпретирует силу коэффициента корреляции."""
    abs_corr = abs(corr_coef)
    if abs_corr < 0.1:
        return "очень слабая"
    elif abs_corr < 0.3:
        return "слабая"
    elif abs_corr < 0.5:
        return "умеренная"
    elif abs_corr < 0.7:
        return "сильная"
    else:
        return "очень сильная"

def interpret_statistical_significance(p_value):
    """Интерпретирует статистическую значимость p-value."""
    if p_value < 0.001:
        return "высокозначимо"
    elif p_value < 0.01:
        return "очень значимо"
    elif p_value < 0.05:
        return "значимо"
    else:
        return "не значимо"

def analyze_corr(df, col1, col2, show_spearman=True):
    """Анализ корреляции между двумя колонками датафрейма."""
    data = df[[col1, col2]].dropna()
    if len(data) < 3:
        print(f"Недостаточно данных для {col1} vs {col2}")
        return
    pearson_corr, pearson_p = stats.pearsonr(data[col1], data[col2])
    spearman_corr, spearman_p = stats.spearmanr(data[col1], data[col2])
    print(f"=== {col1} vs {col2} ===")
    print(f"Наблюдений: {len(data)}")
    print(f"Пирсон: {pearson_corr:.4f} ({interpret_correlation_strength(pearson_corr)})")
    print(f"p-value: {pearson_p:.6f} [{interpret_statistical_significance(pearson_p)}]")
    print(f"R^2: {pearson_corr**2:.4f} ({pearson_corr**2*100:.1f}% дисперсии)")
    if show_spearman:
        print(f"Спирмен: {spearman_corr:.4f} ({interpret_correlation_strength(spearman_corr)})")
        print(f"p-value: {spearman_p:.6f} [{interpret_statistical_significance(spearman_p)}]")
    print()

def plot_distributions(df, features, group_name=""):
    """Строит гистограммы распределений для каждого признака."""
    n_cols = 4
    n_rows = int(np.ceil(len(features) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 3 * n_rows))
    axes = axes.flatten()
    for i, feat in enumerate(features):
        ax = axes[i]
        sns.histplot(df[feat], kde=True, ax=ax)
        ax.set_title(f"{feat}\nSkewness: {df[feat].skew():.2f}", fontsize=9)
        ax.set_xlabel('')
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    plt.suptitle(f'Distributions - {group_name}', fontsize=18, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

def plot_boxplots(df, features, group_name=""):
    """Строит box plots для обнаружения выбросов."""
    n_cols = 4
    n_rows = int(np.ceil(len(features) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 3 * n_rows))
    axes = axes.flatten()
    for i, feat in enumerate(features):
        ax = axes[i]
        sns.boxplot(y=df[feat], ax=ax)
        ax.set_title(feat, fontsize=9)
        ax.set_ylabel('')
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    plt.suptitle(f'Box Plots (Outliers) - {group_name}', fontsize=18, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

def plot_correlation_heatmap(df, features, targets=None, group_name=""):
    """Строит тепловую карту корреляций признаков и таргетов."""
    if targets is None:
        targets = []
    corr_cols = features + targets
    corr = df[corr_cols].corr()
    plt.figure(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                linewidths=0.5, square=True, cbar_kws={'shrink': 0.8},
                annot_kws={'size': 8})
    plt.title(f'Correlation Matrix - {group_name}', fontsize=14, fontweight='bold', y=1.02)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    return corr

def print_top_correlations_with_targets(df, features, targets):
    """Выводит топ-5 признаков по корреляции с каждым таргетом."""
    corr_matrix = df[features + targets].corr()
    for target in targets:
        other_targets = [t for t in targets if t != target]
        corr_target = corr_matrix[target].drop(other_targets + [target], errors='ignore')
        corr_target = corr_target.sort_values(ascending=False)
        print(f"\n{'='*50}")
        print(f"Топ-5 корреляций с {target}:")
        print('='*50)
        for feat, corr_val in corr_target.head(5).items():
            strength = interpret_correlation_strength(corr_val)
            print(f"  {feat}: {corr_val:.4f} ({strength})")

def find_constant_features(df, features):
    """Находит константные признаки (нулевая дисперсия)."""
    return [f for f in features if df[f].std() == 0]

def find_low_variance_features(df, features, threshold=0.01):
    """Находит почти константные признаки с дисперсией ниже порога."""
    sel = VarianceThreshold(threshold=threshold)
    sel.fit(df[features])
    low_var_feats = [features[i] for i in np.where(~sel.get_support())[0]]
    return low_var_feats

def find_highly_correlated_pairs(df, features, threshold=0.95):
    """Находит пары признаков с корреляцией выше порога."""
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    high_corr_pairs = []
    for col in upper.columns:
        for idx in upper.index:
            if upper.loc[col, idx] > threshold:
                high_corr_pairs.append({
                    'feature1': col,
                    'feature2': idx,
                    'correlation': upper.loc[col, idx]
                })
    return pd.DataFrame(high_corr_pairs).sort_values('correlation', ascending=False)

def detect_drift(train_df, test_df, features, threshold=0.05):
    """Обнаруживает дрифт распределений между train и test."""
    drift_results = []
    for feat in features:
        ks_stat, p_value = stats.ks_2samp(train_df[feat], test_df[feat])
        drift_results.append({
            'feature': feat,
            'ks_statistic': ks_stat,
            'p_value': p_value,
            'drift': p_value < threshold
        })
    return pd.DataFrame(drift_results)

def analyze_outliers(df, features):
    """Анализирует выбросы методом IQR."""
    outlier_report = []
    for feat in features:
        Q1 = df[feat].quantile(0.25)
        Q3 = df[feat].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = df[(df[feat] < lower_bound) | (df[feat] > upper_bound)]
        outlier_pct = len(outliers) / len(df) * 100
        outlier_report.append({
            'feature': feat,
            'outliers_count': len(outliers),
            'outliers_pct': outlier_pct
        })
    return pd.DataFrame(outlier_report).sort_values('outliers_pct', ascending=False)

## Анализ группы 2: Электронные и зарядовые характеристики




#### Определение списка признаков группы

In [ ]:
ELECTRONIC_GROUP_NAME = 'Электронные и зарядовые характеристики'
ELECTRONIC_FEATURES = [
    'MaxPartialCharge', 'MinPartialCharge', 'MaxAbsPartialCharge', 'MinAbsPartialCharge',
    'PEOE_VSA1', 'PEOE_VSA2', 'PEOE_VSA3', 'PEOE_VSA4', 'PEOE_VSA5',
    'PEOE_VSA6', 'PEOE_VSA7', 'PEOE_VSA8', 'PEOE_VSA9', 'PEOE_VSA10',
    'PEOE_VSA11', 'PEOE_VSA12', 'PEOE_VSA13', 'PEOE_VSA14'
]

# Первичная информация о признаках
display_columns_info(train, ELECTRONIC_FEATURES)

**Наблюдения:**
- Пропуски обнаружены в четырёх признаках: `MaxPartialCharge`, `MinPartialCharge`, `MaxAbsPartialCharge`, `MinAbsPartialCharge` (по 2 пропуска).
- Признаки `MinPartialCharge` и `MaxAbsPartialCharge` имеют очень низкую дисперсию и подозрительно малое количество уникальных значений.

### Обработка пропусков

In [ ]:
# Заполним пропуски медианой в зарядовых признаках
charge_cols_with_missing = ['MaxPartialCharge', 'MinPartialCharge', 'MaxAbsPartialCharge', 'MinAbsPartialCharge']
for col in charge_cols_with_missing:
    median_val = train[col].median()
    train[col].fillna(median_val, inplace=True)
print("Пропуски заполнены медианами.")

## Поиск константных и почти константных признаков



In [ ]:
constant = find_constant_features(train, ELECTRONIC_FEATURES)
print(f"Константные признаки (нулевая дисперсия): {constant}")

low_var = find_low_variance_features(train, ELECTRONIC_FEATURES, threshold=0.01)
print(f"Почти константные признаки (дисперсия < 0.01): {low_var}")

**Результат:**
- Константных признаков нет.
- Почти константные: `MinPartialCharge`, `MaxAbsPartialCharge`. Они демонстрируют очень низкую вариативность и практически дублируют друг друга (r ≈ -0.93). Их следует удалить.

## Анализ распределений и выбросов


In [ ]:
plot_distributions(train, ELECTRONIC_FEATURES, group_name=ELECTRONIC_GROUP_NAME)
plot_boxplots(train, ELECTRONIC_FEATURES, group_name=ELECTRONIC_GROUP_NAME)

print("АНАЛИЗ ВЫБРОСОВ (метод IQR)")
display(analyze_outliers(train, ELECTRONIC_FEATURES))

**Характеристика распределений:**
- Зарядовые признаки (`MaxPartialCharge`, `MinAbsPartialCharge`) распределены относительно симметрично.
- Признаки `PEOE_VSA*` сильно скошены вправо (много нулевых значений, единичные экстремальные выбросы).
- Наибольшее количество выбросов: `PEOE_VSA4` (155), `PEOE_VSA13` (131), `PEOE_VSA5` (118). Удалять их не следует, но необходимо робастное масштабирование для линейных моделей.

### Корреляционный анализ

In [ ]:
corr_matrix = plot_correlation_heatmap(train, ELECTRONIC_FEATURES, TARGETS, group_name=ELECTRONIC_GROUP_NAME)

high_corr_pairs = find_highly_correlated_pairs(train, ELECTRONIC_FEATURES, threshold=0.95)
print("ПАРЫ ПРИЗНАКОВ С КОРРЕЛЯЦИЕЙ > 0.95")
display(high_corr_pairs)

**Выявленные сильно скоррелированные пары:**
- `MaxPartialCharge` и `MinAbsPartialCharge` (0.975) – информация дублируется.
- `MinPartialCharge` и `MaxAbsPartialCharge` (-0.93) – также почти линейная зависимость.

Один признак из каждой пары должен быть удалён, чтобы избежать мультиколлинеарности.

 ### Анализ дрифта между train и test

In [ ]:
drift_report = detect_drift(train, test, ELECTRONIC_FEATURES)
drift_detected = drift_report[drift_report['drift'] == True]
if len(drift_detected) == 0:
    print("Дрифт распределений между train и test отсутствует.")
else:
    print("Обнаружен дрифт у следующих признаков:")
    display(drift_detected)

### Топ корреляций с целевыми переменными

In [ ]:
print_top_correlations_with_targets(train, ELECTRONIC_FEATURES, TARGETS)

# Удаление избыточных и проблемных признаков

На основе проведённого анализа удаляем следующие признаки:
- `MinAbsPartialCharge` – дублирует `MaxPartialCharge` (r = 0.975).
- `MinPartialCharge` – почти константный, дублируется с `MaxAbsPartialCharge`.
- `MaxAbsPartialCharge` – почти константный, дублирует `MinPartialCharge`.

 ### Удаление выбранных признаков

In [ ]:
ELECTRONIC_FEATURES_TO_DROP = ['MinAbsPartialCharge', 'MinPartialCharge', 'MaxAbsPartialCharge']

train = train.drop(columns=ELECTRONIC_FEATURES_TO_DROP)
test = test.drop(columns=ELECTRONIC_FEATURES_TO_DROP)
ELECTRONIC_FEATURES = [f for f in ELECTRONIC_FEATURES if f not in ELECTRONIC_FEATURES_TO_DROP]

print(f"Оставшиеся признаки ({len(ELECTRONIC_FEATURES)}): {ELECTRONIC_FEATURES}")

# Кластеризация и создание дополнительных дамми-признако

### Кластеризация по PEOE_VSA

In [ ]:
# Выбираем только PEOE_VSA (исключая MaxPartialCharge)
peoe_features = [f for f in ELECTRONIC_FEATURES if f.startswith('PEOE')]

# Масштабируем для кластеризации
scaler_for_cluster = StandardScaler()
train_peoe_scaled = scaler_for_cluster.fit_transform(train[peoe_features])
test_peoe_scaled = scaler_for_cluster.transform(test[peoe_features])

# Обучаем KMeans на train (3 кластера)
kmeans = KMeans(n_clusters=3, random_state=SEED, n_init='auto')
train['peoe_cluster'] = kmeans.fit_predict(train_peoe_scaled)
test['peoe_cluster'] = kmeans.predict(test_peoe_scaled)

print("Распределение по кластерам в train:")
print(train['peoe_cluster'].value_counts())

# Средние значения таргетов по кластерам
print("\nСредние IC50, CC50, SI по кластерам:")
display(train.groupby('peoe_cluster')[TARGETS].mean())

**Наблюдение:** Кластер 2 имеет значительно более высокий средний SI, что говорит о его особом «электронном профиле», связанном с селективностью.

### ANOVA для проверки значимости кластеров

In [ ]:
print("ANOVA: проверка различий средних значений таргетов по кластерам")
for target in TARGETS:
    groups = [train[train['peoe_cluster'] == i][target] for i in range(3)]
    f_stat, p_val = f_oneway(*groups)
    significance = interpret_statistical_significance(p_val)
    print(f"{target}: F = {f_stat:.2f}, p-value = {p_val:.4f} ({significance})")

**Результат ANOVA:**
- `CC50` – высокозначимо (p ≈ 0.0001), `SI` – очень значимо (p ≈ 0.0091), `IC50` – не значимо (p ≈ 0.0514).
- Значит, кластеры действительно разделяют молекулы по цитотоксичности и селективности.

## Создание дамми-переменных для кластеров

In [ ]:
encoder = OneHotEncoder(sparse_output=False)

# Обучаем на train
cluster_train = encoder.fit_transform(train[['peoe_cluster']])
cluster_test = encoder.transform(test[['peoe_cluster']])
cluster_cols = [f'cluster_{i}' for i in range(cluster_train.shape[1])]

# Добавляем дамми-столбцы к train и test
train_cluster_df = pd.DataFrame(cluster_train, columns=cluster_cols, index=train.index)
test_cluster_df = pd.DataFrame(cluster_test, columns=cluster_cols, index=test.index)

train = pd.concat([train, train_cluster_df], axis=1)
test = pd.concat([test, test_cluster_df], axis=1)

print(f"Добавлены дамми-признаки: {cluster_cols}")

В наборе признаков появились три бинарные колонки `cluster_0`, `cluster_1`, `cluster_2`.
Они будут использоваться на этапе моделирования как дополнительный источник информации.

## Выводы по группе «Электронные и зарядовые характеристики»

### Качество данных
| Показатель | Результат |
|------------|-----------|
| Пропуски | В четырёх зарядовых дескрипторах (`MaxPartialCharge`, `MinPartialCharge`, `MaxAbsPartialCharge`, `MinAbsPartialCharge`) – по 2 пропуска (0.27%). Заполнены медианами. |
| Выбросы | Наибольшее число выбросов (IQR) у `PEOE_VSA4` (155), `PEOE_VSA13` (131) и `PEOE_VSA5` (118). Не удаляются, требуют робастного масштабирования. |
| Константные / почти константные | `MinPartialCharge` и `MaxAbsPartialCharge` (дисперсия < 0.01, зеркальная корреляция –0.93). Удалены. |
| Дублирующие признаки | `MinAbsPartialCharge` дублирует `MaxPartialCharge` (r = 0.975). Удалён. |

### Необходимые преобразования
| Преобразование | Признаки | Причина |
|---------------|----------|---------|
| Винзоризация (1‑й и 99‑й перцентили) | Все непрерывные (`MaxPartialCharge` и 14 `PEOE_VSA*`) | Наличие экстремальных выбросов |
| RobustScaler | Все непрерывные (после винзоризации) | Защита линейных моделей от выбросов |
| Удаление | `MinAbsPartialCharge`, `MinPartialCharge`, `MaxAbsPartialCharge` | Мультиколлинеарность / низкая вариабельность |
| Добавление признаков | Кластерные дамми `cluster_0`, `cluster_1`, `cluster_2` | ANOVA показала высокую значимость различий между кластерами для CC50 (p=0.0001) и SI (p=0.0091) |

### Наиболее коррелирующие с таргетами признаки
| Таргет | Топ-5 признаков (корреляция Пирсона) |
|--------|-------------------------------------|
| IC50 | `PEOE_VSA7` (–0.22), `PEOE_VSA6` (–0.18), `MaxPartialCharge` (0.13), `PEOE_VSA4` (0.09), `PEOE_VSA14` (0.08) |
| CC50 | `PEOE_VSA7` (–0.24), `PEOE_VSA6` (–0.19), `PEOE_VSA1` (–0.14), `MaxAbsPartialCharge` (–0.13), `MinPartialCharge` (0.12) |
| SI | Все корреляции очень слабые (|r| < 0.07). Основной сигнал — через кластеры: кластер 2 имеет средний SI ≈ 241. |

### Дополнительные замечания
- Линейная связь электронных дескрипторов с активностью слабая. Ни логарифмирование, ни винзоризация, ни взаимодействия признаков не превращают их в сильные линейные предикторы.
- Кластерный анализ (KMeans, 3 кластера на основе PEOE_VSA) выявил группы молекул, значимо различающиеся по CC50 и SI. Это доказывает, что группа несёт полезную информацию, особенно для предсказания токсичности и селективности.
- Рекомендуется использовать **нелинейные модели** (градиентный бустинг, Random Forest) с полным набором непрерывных признаков и кластерными дамми.
- Для линейных моделей обязательна предварительная винзоризация + RobustScaler. Кластерные дамми также могут быть добавлены.
- **Итоговый набор признаков:** 15 непрерывных дескрипторов + 3 дамми кластеров = **18 признаков**, полностью готовых к моделированию.

 ## Финальная подготовка признаков для моделирования

In [ ]:
# Используем train и test, которые уже очищены:
# - заполнены пропуски
# - удалены MinAbsPartialCharge, MinPartialCharge, MaxAbsPartialCharge
# - добавлены cluster_0, cluster_1, cluster_2

id_col = 'index'
target_cols = ['IC50', 'CC50', 'SI']
cluster_cols = ['cluster_0', 'cluster_1', 'cluster_2']

# Список оставшихся непрерывных признаков группы 2
group2_cont_features = [
    'MaxPartialCharge',
    'PEOE_VSA1', 'PEOE_VSA2', 'PEOE_VSA3', 'PEOE_VSA4', 'PEOE_VSA5',
    'PEOE_VSA6', 'PEOE_VSA7', 'PEOE_VSA8', 'PEOE_VSA9', 'PEOE_VSA10',
    'PEOE_VSA11', 'PEOE_VSA12', 'PEOE_VSA13', 'PEOE_VSA14'
]

# Все признаки группы 2 (непрерывные + дамми)
group2_feature_cols = group2_cont_features + cluster_cols

# Итоговые датафреймы
X_train_group2 = train[[id_col] + group2_feature_cols].copy()
X_test_group2 = test[[id_col] + group2_feature_cols].copy()
y_group2 = train[target_cols].copy()

print("Размерности:")
print(f"X_train_group2: {X_train_group2.shape}")
print(f"X_test_group2 : {X_test_group2.shape}")
print(f"y_group2      : {y_group2.shape}")
print("\nПризнаки:", list(X_train_group2.columns))

## Итоговые признаки после EDA (Группа 2)

**Состояние данных:**
- Пропуски заполнены медианами (зарядовые дескрипторы, 2 молекулы).
- Удалены дублирующие и почти константные признаки:  
  `MinAbsPartialCharge`, `MinPartialCharge`, `MaxAbsPartialCharge`.
- Добавлены кластерные дамми-признаки:  
  `cluster_0`, `cluster_1`, `cluster_2` (результат KMeans на основе PEOE_VSA, обоснованы ANOVA).

**Итоговый набор признаков (18 штук):**
- Непрерывные (15):  
  `MaxPartialCharge`, `PEOE_VSA1` … `PEOE_VSA14`.
- Бинарные дамми (3):  
  `cluster_0`, `cluster_1`, `cluster_2`.

**Переменные для моделирования:**
| Переменная | Размер | Содержание |
|-----------|--------|------------|
| `X_train_group2` | (751, 19) | `index` + 18 признаков |
| `X_test_group2`  | (250, 19) | `index` + 18 признаков |
| `y_group2`       | (751, 3)  | Таргеты `IC50`, `CC50`, `SI` (исходная шкала) |

**Примечание:**  
Масштабирование, винзоризация и логарифмирование будут выполнены централизованно  
для всех групп при объединении датасета перед этапом моделирования.